# Mid-Circuit Measurements with Pauli Propagation

This notebook shows how to handle a circuit with a **mid-circuit measurement and feed-forward** using the Pauli Propagator.

In [1]:
import pennylane as qml
import numpy as np
from pprop import Propagator

/home/samonaco/Pauli-Propagator/.venv/lib/python3.12/site-packages/pennylane/operation.py:2622: PennyLaneDeprecationWarning: Observable is deprecated and will be removed in v0.43. A generic Operator class should be used instead. If defining an Operator, set the is_hermitian property to True. If checking if an Operator is Hermitian, check the is_hermitian property. 
  warnings.warn(


### Circuit definitions

We use 4 qubits. The ansatz is two layers of RX + a chain of CNOTs.

In [2]:
NUM_QUBITS = 4
N = NUM_QUBITS - 1  # index of the last qubit (the one we measure at the end)


def ansatz(num_qubits, params):
    """Ansatz: RX on each qubit, CNOT chain, RX on each qubit."""
    param_idx = 0
    for qubit in range(num_qubits):
        qml.RX(params[param_idx], qubit)
        param_idx += 1
    for qubit in range(num_qubits - 1):
        qml.CNOT(wires=[qubit, qubit + 1])
    for qubit in range(num_qubits):
        qml.RX(params[param_idx], qubit)
        param_idx += 1
    return param_idx  # next free parameter index

In [3]:
# Full circuit: ansatz -> measure q0 -> conditional RX or RY on q_{n-1} -> Z
def full_circuit(num_qubits, params):
    param_idx = ansatz(num_qubits, params)
    m = qml.measure(0)
    qml.cond(m == 0, qml.RX)(params[param_idx],     wires=num_qubits - 1)
    qml.cond(m == 1, qml.RY)(params[param_idx + 1], wires=num_qubits - 1)
    return qml.expval(qml.PauliZ(num_qubits - 1))


# Branch 0: postselect qubit 0 on |0>, then apply RX
def branch0_circuit(num_qubits, params):
    param_idx = ansatz(num_qubits, params)
    qml.measure(0, postselect=0)  # project qubit 0 onto |0>
    qml.RX(params[param_idx], wires=num_qubits - 1)
    return qml.expval(qml.PauliZ(num_qubits - 1))


# Branch 1: postselect qubit 0 on |1>, then apply RY
def branch1_circuit(num_qubits, params):
    param_idx = ansatz(num_qubits, params)
    qml.measure(0, postselect=1)  # project qubit 0 onto |1>
    qml.RY(params[param_idx + 1], wires=num_qubits - 1)
    return qml.expval(qml.PauliZ(num_qubits - 1))


# Just the ansatz measuring Z_0, to get p(0) and p(1)
def z0_circuit(num_qubits, params):
    ansatz(num_qubits, params)
    return qml.expval(qml.PauliZ(0))


dev = qml.device("default.qubit", wires=NUM_QUBITS)
qnode_full    = qml.QNode(full_circuit,    dev)
qnode_branch0 = qml.QNode(branch0_circuit, dev)
qnode_branch1 = qml.QNode(branch1_circuit, dev)
qnode_z0      = qml.QNode(z0_circuit,      dev)

print(qml.draw(qnode_full)(NUM_QUBITS, np.zeros(100)))

0: ──RX(0.00)─╭●──RX(0.00)──────────────────────┤↗├─────────────────────┤     
1: ──RX(0.00)─╰X─╭●─────────RX(0.00)─────────────║──────────────────────┤     
2: ──RX(0.00)────╰X────────╭●─────────RX(0.00)───║──────────────────────┤     
3: ──RX(0.00)──────────────╰X─────────RX(0.00)───║───RX(0.00)──RY(0.00)─┤  <Z>
                                                 ╚═══╩═════════╝              


## Pauli Propagator approach

1. Consider the two circuits separately depending on the two measurements
of qubit 0 `branch0_circuit` and `branch1_circuit`

2. Propagate right after where the split happends:
    * **branch 0**: $\mathcal{B}_0 = RX_8^\dagger\, Z_3\, RX_8 = \cos\theta_8 Z_3 + \sin\theta_8 Y_3$
    * **branch 1**: $\mathcal{B}_1 = RY_9^\dagger\, Z_3\, RY_9 = \cos\theta_9 Z_3 - \sin\theta_9 X_3$
    
3. The observable that needs to be propagated is:
$$\mathcal{O} = \Pi_0 \otimes \mathcal{B}_0 + \Pi_1 \otimes \mathcal{B}_1$$

where $\Pi$ are the projectors:

$$\Pi_0 = \frac{I_0 + Z_0}{2}\qquad\qquad \Pi_1 = \frac{I_0 - Z_0}{2}$$

expanding the observable $\mathcal{O}$:

$$
\begin{align*}
    O &= +\frac{\cos\theta_8 + \cos\theta_9}{2} Z_3 + \frac{\cos\theta_8 - \cos\theta_9}{2} Z_0Z_3 + \\
        &\hphantom{=} +\frac{\sin\theta_8}{2}Y_3 + \frac{\sin\theta_8}{2}Z_0Y_3 - \frac{\sin\theta_9}{2}X_3 + \frac{\sin\theta_9}{2}Z_0X_3
\end{align*}
$$

and it needs to be propagated through the rest of the circuit


In [4]:
# Ansatz for the Propagator: same gates, but returns 6 Pauli observables
def ansatz_multi(params):
    param_idx = 0
    for qubit in range(NUM_QUBITS):
        qml.RX(params[param_idx], qubit)
        param_idx += 1
    for qubit in range(NUM_QUBITS - 1):
        qml.CNOT(wires=[qubit, qubit + 1])
    for qubit in range(NUM_QUBITS):
        qml.RX(params[param_idx], qubit)
        param_idx += 1
    return [
        qml.expval(qml.PauliZ(N)),
        qml.expval(qml.PauliZ(0) @ qml.PauliZ(N)),
        qml.expval(qml.PauliY(N)),
        qml.expval(qml.PauliZ(0) @ qml.PauliY(N)),
        qml.expval(qml.PauliX(N)),
        qml.expval(qml.PauliZ(0) @ qml.PauliX(N)),
    ]

prop = Propagator(ansatz_multi)
prop.propagate()
prop.show()

0: ──RX(0.00)─╭●──RX(4.00)─────────────────────┤      ╭<Z@Z>      ╭<Z@Y>      ╭<Z@X>
1: ──RX(1.00)─╰X─╭●─────────RX(5.00)───────────┤      │           │           │     
2: ──RX(2.00)────╰X────────╭●─────────RX(6.00)─┤      │           │           │     
3: ──RX(3.00)──────────────╰X─────────RX(7.00)─┤  <Z> ╰<Z@Z>  <Y> ╰<Z@Y>  <X> ╰<Z@X>


In [5]:
def mcm_propagator(params, theta_a, theta_b):
    """
    Compute <Z>_full for the MCM circuit using the Pauli Propagator.

    params  : ansatz parameters (length 2*NUM_QUBITS)
    theta_a : angle of the RX gate applied when outcome = 0
    theta_b : angle of the RY gate applied when outcome = 1
    """
    ca, sa = np.cos(theta_a), np.sin(theta_a)
    cb, sb = np.cos(theta_b), np.sin(theta_b)
    v = prop(params)   # [<Z_N>, <Z_0 Z_N>, <Y_N>, <Z_0 Y_N>, <X_N>, <Z_0 X_N>]
    return (
        (ca + cb) / 2 * v[0]   # <Z_{n-1}>
      + (ca - cb) / 2 * v[1]   # <Z_0 Z_{n-1}>
      +  sa / 2       * v[2]   # <Y_{n-1}>
      +  sa / 2       * v[3]   # <Z_0 Y_{n-1}>
      -  sb / 2       * v[4]   # <X_{n-1}>
      +  sb / 2       * v[5]   # <Z_0 X_{n-1}>
    )

## Verification: Propagator vs full PennyLane circuit

In [6]:
# The ansatz uses 2*NUM_QUBITS parameters; theta_a and theta_b are separate
NUM_ANSATZ_PARAMS = 2 * NUM_QUBITS

np.random.seed(0)
print(f"{'Trial':>5}  {'PennyLane full':>14}  {'Propagator':>12}  {'match':>6}")
print("-" * 50)
for trial in range(10):
    all_params = np.random.rand(100)
    ansatz_params = all_params[:NUM_ANSATZ_PARAMS]
    theta_a = all_params[NUM_ANSATZ_PARAMS]
    theta_b = all_params[NUM_ANSATZ_PARAMS + 1]

    full = float(qnode_full(NUM_QUBITS, all_params))
    pprop_val = float(mcm_propagator(ansatz_params, theta_a, theta_b))

    print(f"{trial:>5}  {full:>14.8f}  {pprop_val:>12.8f}  {str(np.isclose(full, pprop_val)):>6}")

Trial  PennyLane full    Propagator   match
--------------------------------------------------
    0     -0.40702806   -0.40702806    True
    1     -0.11742520   -0.11742520    True
    2      0.26295013    0.26295013    True
    3      0.23723922    0.23723922    True
    4     -0.07115199   -0.07115199    True
    5      0.38024129    0.38024129    True
    6      0.50885587    0.50885587    True
    7      0.89104646    0.89104646    True
    8      0.02608178    0.02608178    True
    9      0.53497053    0.53497053    True
